# CNN-EXP-049 — Colab/Kaggle training launcher

Restartable CUDA workflow for the R16 coarse-to-fine cascade. Run `SETUP`, `INSPECT`, and `SMOKE` first. Network A is trained one OOF fold per session; all folds write to the same persistent run directory.

In [ ]:
from pathlib import Path
import hashlib, os, shutil, subprocess, sys, tarfile

IS_KAGGLE = Path('/kaggle').exists()
IS_COLAB = 'google.colab' in sys.modules
if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

# EDIT THESE PATHS. Kaggle paths refer to attached Datasets; Colab paths refer to Drive.
if IS_KAGGLE:
    CODE_BUNDLE = Path('/kaggle/input/epic-exp049-code/epic-exp049-code.zip')
    DATA_ARCHIVE = Path('/kaggle/input/epic-nematostella/jaNemVect1.1.tgz')
    RISK_BUNDLE = Path('/kaggle/input/epic-exp047-risk-cache/exp047-risk-cache.zip')
    # Keep temporary code/data/cache out of notebook outputs; export only one ZIP.
    WORK_ROOT = Path('/tmp/epic_exp049')
    PERSIST_ROOT = Path('/tmp/exp049_run')
    # Attach the previous notebook output and edit this path when resuming.
    RESUME_BUNDLE = Path('/kaggle/input/EDIT-PREVIOUS-OUTPUT/exp049_run.zip')
    EXPORT_BUNDLE = Path('/kaggle/working/exp049_run.zip')
else:
    CODE_BUNDLE = Path('/content/drive/MyDrive/EPIC/epic-exp049-code.zip')
    DATA_ARCHIVE = Path('/content/drive/MyDrive/EPIC/jaNemVect1.1.tgz')
    RISK_BUNDLE = Path('/content/drive/MyDrive/EPIC/exp047-risk-cache.zip')
    WORK_ROOT = Path('/content/epic_exp049')
    # Train/read memmaps on the VM SSD; only the compressed resume file lives on Drive.
    PERSIST_ROOT = Path('/content/exp049_run')
    RESUME_BUNDLE = Path('/content/drive/MyDrive/EPIC/exp049_run.zip')
    EXPORT_BUNDLE = RESUME_BUNDLE

FOLDS = 3
FOLD_TO_RUN = 0       # change to 1 and 2 in later sessions
A_STEPS = 20000
B_STEPS = 50000
MAX_MINUTES = 300     # graceful checkpoint before a typical session limit
FILM_BLOCKS = '2,4,6,8'
EXPECTED_MD5 = '733dccb6393a3c07b6e0d7ffaed3686f'
print('platform:', 'kaggle' if IS_KAGGLE else 'colab/local')
print('bundle:', CODE_BUNDLE)
print('data:', DATA_ARCHIVE)
print('EXP047 risk bundle:', RISK_BUNDLE, '(optional)')
print('local run:', PERSIST_ROOT)
print('resume bundle:', RESUME_BUNDLE, '(used only when it exists)')
print('export bundle:', EXPORT_BUNDLE)

## SETUP
Locate and prepare code, Nematostella data, and risk cache. Kaggle may expose either the original archives or their automatically expanded contents; both layouts are supported. This cell does not overwrite the persistent run directory.

In [ ]:
KAGGLE_INPUT = Path('/kaggle/input')
KAGGLE_DATASETS = KAGGLE_INPUT / 'datasets'
def discover_file(configured, filename):
    if configured.is_file():
        return configured
    matches = list(KAGGLE_DATASETS.rglob(filename)) if IS_KAGGLE else []
    if len(matches) > 1:
        raise RuntimeError(f'multiple {filename} files: {matches}')
    return matches[0] if matches else configured
CODE_BUNDLE = discover_file(CODE_BUNDLE, 'epic-exp049-code.zip')
DATA_ARCHIVE = discover_file(DATA_ARCHIVE, 'jaNemVect1.1.tgz')
RISK_BUNDLE = discover_file(RISK_BUNDLE, 'exp047-risk-cache.zip')
WORK_ROOT.mkdir(parents=True, exist_ok=True)
if CODE_BUNDLE.is_file():
    shutil.unpack_archive(str(CODE_BUNDLE), str(WORK_ROOT))
else:
    # Kaggle normally expands uploaded ZIP datasets automatically.
    input_runners = list(KAGGLE_DATASETS.rglob('scripts/run_cnn_exp049_cloud.py'))
    assert len(input_runners) == 1, ('code bundle/runner not found', input_runners)
    source_project = input_runners[0].parents[1]
    shutil.copytree(source_project, WORK_ROOT / 'project', dirs_exist_ok=True)
if RISK_BUNDLE.is_file():
    shutil.unpack_archive(str(RISK_BUNDLE), str(WORK_ROOT))
runner_matches = list(WORK_ROOT.rglob('scripts/run_cnn_exp049_cloud.py'))
assert len(runner_matches) == 1, runner_matches
PROJECT_ROOT = runner_matches[0].parents[1]
unpacked = PROJECT_ROOT / 'data/raw/unpacked'
assembly_dir = unpacked / 'jaNemVect1.1'
if not assembly_dir.is_dir():
    unpacked.mkdir(parents=True, exist_ok=True)
    genomes = list(KAGGLE_DATASETS.rglob('jaNemVect1.1/genome/genome.fa.gz')) if IS_KAGGLE else []
    if len(genomes) == 1:
        # Read the Kaggle Dataset in place; do not copy hundreds of MB per session.
        assembly_dir.symlink_to(genomes[0].parents[1], target_is_directory=True)
    elif len(genomes) > 1:
        raise RuntimeError(f'multiple Nematostella assemblies: {genomes}')
    elif DATA_ARCHIVE.is_file():
        md5 = hashlib.md5()
        with DATA_ARCHIVE.open('rb') as stream:
            for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b''):
                md5.update(chunk)
        digest = md5.hexdigest()
        assert digest == EXPECTED_MD5, (digest, EXPECTED_MD5)
        with tarfile.open(DATA_ARCHIVE, 'r:gz') as archive:
            archive.extractall(unpacked)
    else:
        raise FileNotFoundError('Nematostella archive/directory not found')
assert (assembly_dir / 'genome/genome.fa.gz').is_file(), assembly_dir
if IS_KAGGLE and not RESUME_BUNDLE.is_file():
    resume_matches = list(Path('/kaggle/input/notebooks').rglob('exp049_run.zip'))
    if resume_matches:
        RESUME_BUNDLE = max(resume_matches, key=lambda path: path.stat().st_size)
        print('auto-detected baseline resume:', RESUME_BUNDLE)
PERSIST_ROOT.mkdir(parents=True, exist_ok=True)
if RESUME_BUNDLE.is_file() and not any(PERSIST_ROOT.iterdir()):
    print('restoring previous run from:', RESUME_BUNDLE)
    shutil.unpack_archive(str(RESUME_BUNDLE), str(PERSIST_ROOT))
persistent_processed = PERSIST_ROOT / 'prepared_data'
persistent_processed.mkdir(parents=True, exist_ok=True)
project_processed = PROJECT_ROOT / 'data/processed'
if not project_processed.exists() and not project_processed.is_symlink():
    project_processed.symlink_to(persistent_processed, target_is_directory=True)
elif project_processed.resolve() != persistent_processed.resolve():
    raise RuntimeError(f'unexpected processed-data path: {project_processed}')
RUNNER = PROJECT_ROOT / 'scripts/run_cnn_exp049_cloud.py'
CACHE_DIR = PERSIST_ROOT / 'regional_cache'
risk_roots = [WORK_ROOT, KAGGLE_DATASETS] if IS_KAGGLE else [WORK_ROOT]
risk_cache_matches = list(dict.fromkeys(p for root in risk_roots for p in root.rglob('train_negative_stratum_u8.dat')))
risk_manifest_matches = list(dict.fromkeys(p for root in risk_roots for p in root.rglob('risk_strata_manifest.json')))
RISK_ARGS = ()
if len(risk_cache_matches) == 1 and len(risk_manifest_matches) == 1:
    RISK_ARGS = ('--risk-cache', risk_cache_matches[0], '--risk-manifest', risk_manifest_matches[0])
print('project:', PROJECT_ROOT)
print('runner:', RUNNER)
print('cache:', CACHE_DIR)
print('risk loss:', 'EXP047 exact weights' if RISK_ARGS else 'base BCE pilot')

In [ ]:
def run_stage(stage, *extra):
    command = [
        sys.executable, '-u', str(RUNNER), stage,
        '--run-dir', str(PERSIST_ROOT),
        '--cache-dir', str(CACHE_DIR),
        '--folds', str(FOLDS),
        '--device', 'cuda',
        *map(str, extra),
    ]
    print('RUN:', subprocess.list2cmdline(command), flush=True)
    process = subprocess.Popen(command, cwd=PROJECT_ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True)
    code = process.wait()
    if code:
        raise RuntimeError(f'stage {stage} failed with exit code {code}')

## INSPECT + SMOKE

In [ ]:
run_stage('inspect')
run_stage('smoke', '--film-blocks', FILM_BLOCKS)

## OOF NETWORK A
Run the following cell for fold 0, then change `FOLD_TO_RUN` to 1 and 2 in later sessions. Resume is automatic. If `MAX_MINUTES` stops training early, `cache-a` will intentionally refuse the incomplete checkpoint; rerun the same cell until the fold reaches `A_STEPS`.

In [ ]:
run_stage(
    'train-a', '--fold', FOLD_TO_RUN, '--a-steps', A_STEPS, '--a-batch', 2,
    '--max-minutes', MAX_MINUTES, '--save-every', 500,
)
run_stage('cache-a', '--fold', FOLD_TO_RUN, '--cache-part', 'train')

## EXP049 A-v1 COUNT-AWARE PILOT
Run this section instead of the legacy B section. It preserves A-v0, trains only fold 0 with a supervised latent bottleneck and regional Poisson+ranking loss, and saves held-out metrics at 5K/10K/20K.

In [ ]:
RATE_RUNNER = PROJECT_ROOT / 'scripts/run_cnn_exp049a_rate_pilot.py'
assert RATE_RUNNER.is_file(), RATE_RUNNER
RATE_RUN_ROOT = Path('/tmp/exp049a_rate_pilot') if IS_KAGGLE else Path('/content/exp049a_rate_pilot')
RATE_EXPORT = Path('/kaggle/working/exp049a_rate_pilot.zip') if IS_KAGGLE else Path('/content/drive/MyDrive/EPIC/exp049a_rate_pilot.zip')
rate_resume_matches = list(Path('/kaggle/input/notebooks').rglob('exp049a_rate_pilot.zip')) if IS_KAGGLE else ([RATE_EXPORT] if RATE_EXPORT.is_file() else [])
assert len(rate_resume_matches) <= 1, rate_resume_matches
RATE_RUN_ROOT.mkdir(parents=True, exist_ok=True)
if rate_resume_matches and not any(RATE_RUN_ROOT.iterdir()):
    print('restoring A-v1 pilot:', rate_resume_matches[0])
    shutil.unpack_archive(str(rate_resume_matches[0]), str(RATE_RUN_ROOT))
def run_rate(stage, *extra):
    command = [sys.executable, '-u', str(RATE_RUNNER), stage, '--run-dir', str(RATE_RUN_ROOT), '--folds', '3', '--fold', '0', '--device', 'cuda', *map(str, extra)]
    print('RUN:', subprocess.list2cmdline(command), flush=True)
    process = subprocess.Popen(command, cwd=PROJECT_ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True)
    code = process.wait()
    if code:
        raise RuntimeError(f'A-v1 stage {stage} failed with exit code {code}')
print('A-v1 run:', RATE_RUN_ROOT)

### Comparable A-v0 fold-0 baseline
This reads the archived binary cache only; it does not modify its predictions.

In [ ]:
run_rate('evaluate-a', '--cache-dir', CACHE_DIR, '--score-link', 'sigmoid')

### Train A-v1 fold 0
Natural R16 sampling is used. If the time limit stops training, export the pilot and rerun this same cell in the next session.

In [ ]:
run_rate(
    'train-a', '--a-steps', 20000, '--a-batch', 2,
    '--a-milestones', '5000,10000,20000', '--lambda-rank', 0.1,
    '--a-positive-fraction', 0.0, '--max-minutes', MAX_MINUTES,
    '--save-every', 500,
)

### Cache and evaluate registered milestones
Run only after training reaches 20K. Each checkpoint is evaluated on the same held-out fold-0 contigs.

In [ ]:
for step in (5000, 10000, 20000):
    cache = RATE_RUN_ROOT / f'cache_step_{step:06d}'
    run_rate('cache-a', '--cache-dir', cache, '--a-checkpoint-step', step)
    run_rate('evaluate-a', '--cache-dir', cache, '--score-link', 'poisson')

### Export A-v1 pilot
Run after every stopped or completed pilot session, then Quick Save with outputs enabled.

In [ ]:
RATE_EXPORT.parent.mkdir(parents=True, exist_ok=True)
rate_temp = RATE_EXPORT.with_name(RATE_EXPORT.stem + '.new').with_suffix('')
rate_archive = Path(shutil.make_archive(str(rate_temp), 'zip', RATE_RUN_ROOT))
os.replace(rate_archive, RATE_EXPORT)
print('saved A-v1 pilot:', RATE_EXPORT)

## EXP049B PAIRED END-TO-END PILOT
Run after A-v1 step 20K exists. This directly compares a zero-conditioned B control against joint A→FiLM→B training for 20K steps. Both arms use the same B initialisation, sampler trajectory, nucleotide loss, and natural validation scan.

In [ ]:
JOINT_RUNNER = PROJECT_ROOT / 'scripts/run_cnn_exp049b_joint_pilot.py'
assert JOINT_RUNNER.is_file(), JOINT_RUNNER
A_INIT = RATE_RUN_ROOT / 'network_a_rate/fold_00/step_020000.pt'
assert A_INIT.is_file(), A_INIT
JOINT_RUN_ROOT = Path('/tmp/exp049b_joint_pilot') if IS_KAGGLE else Path('/content/exp049b_joint_pilot')
JOINT_EXPORT = Path('/kaggle/working/exp049b_joint_pilot.zip') if IS_KAGGLE else Path('/content/drive/MyDrive/EPIC/exp049b_joint_pilot.zip')
joint_resume_matches = list(Path('/kaggle/input/notebooks').rglob('exp049b_joint_pilot.zip')) if IS_KAGGLE else ([JOINT_EXPORT] if JOINT_EXPORT.is_file() else [])
assert len(joint_resume_matches) <= 1, joint_resume_matches
JOINT_RUN_ROOT.mkdir(parents=True, exist_ok=True)
if joint_resume_matches and not any(JOINT_RUN_ROOT.iterdir()):
    print('restoring joint pilot:', joint_resume_matches[0])
    shutil.unpack_archive(str(joint_resume_matches[0]), str(JOINT_RUN_ROOT))
def run_joint(stage, *extra):
    command = [sys.executable, '-u', str(JOINT_RUNNER), stage, '--run-dir', str(JOINT_RUN_ROOT), '--a-checkpoint', str(A_INIT), '--device', 'cuda', *map(str, extra)]
    print('RUN:', subprocess.list2cmdline(command), flush=True)
    process = subprocess.Popen(command, cwd=PROJECT_ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True)
    code = process.wait()
    if code:
        raise RuntimeError(f'joint pilot stage {stage} failed with exit code {code}')
def export_joint():
    JOINT_EXPORT.parent.mkdir(parents=True, exist_ok=True)
    temp = JOINT_EXPORT.with_name(JOINT_EXPORT.stem + '.new').with_suffix('')
    archive = Path(shutil.make_archive(str(temp), 'zip', JOINT_RUN_ROOT))
    os.replace(archive, JOINT_EXPORT)
    print('saved joint pilot:', JOINT_EXPORT)
print('joint run:', JOINT_RUN_ROOT)

### Full-size gradient and memory smoke test
This uses the registered 256-channel B and verifies that nucleotide gradients reach both A and B on the current GPU.

In [ ]:
run_joint('smoke', '--b-channels', 256, '--batch', 2)

### Train the paired 20K arms
Train joint first, export immediately, and Quick Save with outputs. Then train the matched zero-condition control and export again. Resume is automatic from each arm's latest checkpoint.

In [ ]:
run_joint(
    'train-joint', '--steps', 20000, '--batch', 2, '--b-channels', 256,
    '--b-lr', 1e-3, '--a-lr', 1e-4, '--a-aux-weight', 0.1,
    '--max-minutes', MAX_MINUTES, '--save-every', 500, *RISK_ARGS,
)
export_joint()

In [ ]:
run_joint(
    'train-control', '--steps', 20000, '--batch', 2, '--b-channels', 256,
    '--b-lr', 1e-3, '--max-minutes', MAX_MINUTES, '--save-every', 500, *RISK_ARGS,
)
export_joint()

### One matched natural validation scan
Run only after both checkpoints report 20000/20000. The output contains overall and per-contig nucleotide AP plus EPIC Spearman from the same final score.

In [ ]:
run_joint('evaluate-control', '--eval-batch', 1)
run_joint('evaluate-joint', '--eval-batch', 1)
export_joint()

## EXP049B PAIRED CONTINUATION: 20K → 50K
Run only after the paired 20K archive has been restored. Both arms retain their model weights, Adam moments, and sampler state. A new 30K cosine phase uses B LR 3e-4→3e-5 and A LR 3e-5→3e-6.

### Continue joint first
The continuation checkpoint is separate from the immutable 20K checkpoint. Export and Quick Save after every completed or time-limited session.

In [ ]:
run_joint(
    'continue-joint', '--steps', 20000, '--continue-steps', 30000,
    '--batch', 2, '--b-channels', 256, '--b-lr', 1e-3, '--a-lr', 1e-4,
    '--b-continue-lr', 3e-4, '--a-continue-lr', 3e-5,
    '--a-aux-weight', 0.1, '--max-minutes', MAX_MINUTES,
    '--save-every', 500, *RISK_ARGS,
)
export_joint()

### Continue the matched control
Start only after joint has reached 50000/50000 and its archive has been Quick Saved.

In [ ]:
run_joint(
    'continue-control', '--steps', 20000, '--continue-steps', 30000,
    '--batch', 2, '--b-channels', 256, '--b-lr', 1e-3,
    '--b-continue-lr', 3e-4, '--max-minutes', MAX_MINUTES,
    '--save-every', 500, *RISK_ARGS,
)
export_joint()

### Final matched 50K validation
Run once after both continuation checkpoints reach 50000/50000.

In [ ]:
run_joint('evaluate50-control', '--eval-batch', 1)
run_joint('evaluate50-joint', '--eval-batch', 1)
export_joint()

## LEGACY FROZEN-CACHE NETWORK B — DO NOT RUN
Preserved only for the archived A-v0 workflow. The active whole-system experiment is the paired end-to-end pilot above.

In [ ]:
run_stage('evaluate-a')
run_stage(
    'train-b', '--film-blocks', FILM_BLOCKS, '--b-steps', B_STEPS, '--b-batch', 2,
    '--max-minutes', MAX_MINUTES, '--save-every', 500, *RISK_ARGS,
)

## FINAL A + VALIDATION CACHE
Train A on every train contig only after the OOF cache and B training are complete.

In [ ]:
run_stage(
    'train-a', '--fold', -1, '--a-steps', A_STEPS, '--a-batch', 2,
    '--max-minutes', MAX_MINUTES, '--save-every', 500,
)
run_stage('cache-a', '--fold', -1, '--cache-part', 'validation')

## ONE NATURAL VALIDATION PASS

In [ ]:
run_stage('evaluate-b', '--film-blocks', FILM_BLOCKS, '--eval-batch', 2)

## EXPORT / RESUME
Run this after every stopped or completed training session. Colab writes one sequential archive to Drive. Kaggle writes it to notebook output: save a version, attach that output on the next session, and set `RESUME_BUNDLE` to its read-only input path.

In [ ]:
EXPORT_BUNDLE.parent.mkdir(parents=True, exist_ok=True)
temporary_base = EXPORT_BUNDLE.with_name(EXPORT_BUNDLE.stem + '.new').with_suffix('')
archive = Path(shutil.make_archive(str(temporary_base), 'zip', PERSIST_ROOT))
os.replace(archive, EXPORT_BUNDLE)
print('saved resume archive:', EXPORT_BUNDLE)